# Computer Vision – Convolutions Step by Step (Beginner Version)

We will:
1. Load CIFAR-10
2. Flatten + KNN (baseline)
3. Convolution features + KNN
4. Minimal CNN classifier

All code is commented for beginners.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device


## 1. Load CIFAR-10


In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True)
testloader = torch.utils.data.DataLoader(testset, batch_size=128, shuffle=False)


## 2. Flatten + KNN


In [ ]:
subset = 3000
X_train, y_train = [], []

for i in range(subset):
    img, label = trainset[i]
    X_train.append(img.numpy().flatten())
    y_train.append(label)

X_test, y_test = [], []
for i in range(1000):
    img, label = testset[i]
    X_test.append(img.numpy().flatten())
    y_test.append(label)

X_train = np.array(X_train)
X_test = np.array(X_test)

knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train, y_train)
pred = knn.predict(X_test)
print('Flatten + KNN Accuracy:', accuracy_score(y_test, pred))


## 3. Convolution Features + KNN


In [ ]:
class ConvFeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, 8, 3, padding=1)
        self.pool = nn.MaxPool2d(2,2)

    def forward(self, x):
        x = torch.relu(self.conv(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        return x

feature_model = ConvFeatureExtractor().to(device)


## 4. Minimal CNN


In [ ]:
class MinimalCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, 8, 3, padding=1)
        self.pool = nn.MaxPool2d(2,2)
        self.fc = nn.Linear(8*16*16, 10)

    def forward(self, x):
        x = torch.relu(self.conv(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

model = MinimalCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


# Assignment
Modify ONE architectural element and compare results.
Presentation: 5 minutes.


# Rubric (10 points)
- Baseline works (2)
- Conv+KNN works (2)
- Modified CNN works (2)
- Clear comparison (2)
- Explanation (2)
